# Bayesian Optimisation Capstone: Complete Intelligent Workflow

**Course note pointer:** This workflow follows the capstone guidance for how to run the weekly optimisation loop and what to show in the final repository (Module 12 Mini-lesson 12.7 and the Capstone FAQs).

This notebook is my working template for the Bayesian black-box optimisation capstone. The aim is simple: for each of the eight black-box functions, I propose one new input point per week, submit those points in the portal, then fold the returned outputs back into my dataset and repeat.

## What this notebook does:

- ✅ **Intelligent state detection** - skips initial setup if already done
- ✅ **Asks for week number** - processes Week N → generates Week N+1
- ✅ **Auto-loads portal downloads** - handles all formats automatically
- ✅ **Fits Gaussian Process surrogate models** per function
- ✅ **Uses acquisition functions** (EI/UCB/PI/MaxVar) to propose next points
- ✅ **Produces portal-ready strings** (hyphen-separated, six decimals, no spaces)
- ✅ **Displays inputs, results, and plots inline** for review
- ✅ **Creates timestamped backups** before every change
- ✅ **Maintains execution logs** for complete audit trail

## Folder structure:

```text
CapStone_BBO/
├── notebooks/
│   └── CapStone_BBO_Workflow.ipynb
├── data/
│   ├── initial_data/
│   ├── function_1/ ... function_8/
│   │   ├── initial_inputs.npy
│   │   ├── initial_outputs.npy
│   │   └── backups/
│   ├── submissions/
│   ├── week_01/
│   │   ├── Week01-*.txt
│   │   └── outputs.txt
│   ├── week_02/ ...
│   ├── processed/
│   ├── plots/
│   ├── current/
│   └── historical/
│   └── logs/
```

**Course references:**
- Module 12 Mini-lesson 12.7: *A deeper look into the capstone*
- Module 12 Mini-lesson 12.8: *Data and descriptions of functions for Bayesian optimisation competition*
- Module 12 Capstone Project FAQs

---

## Weekly loop (what I actually do)

**Course note pointer:** This is the weekly cadence described in the capstone guidance (Module 12 Mini-lesson 12.7 and the FAQs).

I have eight synthetic black-box functions. Each week I run the same loop:

1. Review the data I already have for each function.
2. Choose one new input point `x` per function using Bayesian Optimisation.
3. Submit each `x` in the portal using the exact numeric string format.
4. Receive the output `y` for each submitted `x`.
5. Append `(x, y)` to the dataset and repeat next week.

I also keep a short note each week covering:
- what method I used and why,
- whether my query was more exploration or exploitation,
- what the latest result taught me,
- how I plan to adjust next week.

**Course references:**
- Module 12 Mini-lesson 12.7: *A deeper look into the capstone*
- Capstone Project FAQs


## Cell 1: Configuration and Setup

**Course note pointer:** The required folder structure and starter files come from the capstone pack described in the FAQs and the Module 12 capstone notes.

In [ ]:
"""
Configuration and Imports

Course note mapping (Module 12):
- Surrogate model (Gaussian Process regression): Mini-lesson 12.1
- Acquisition functions (EI/UCB/PI/max-variance): Mini-lesson 12.6
- Maximisation framing (treating objective as maximise y): Mini-lesson 12.3
"""

from IPython.display import display, Javascript

# This forces the cell output to expand and disables the scrollbar
try:
    display(Javascript("""
        google.colab.output.setIframeHeight(0, true, {maxHeight: 5000})
    """))
except Exception:
    pass

# For standard Jupyter/VS Code:
import pandas as pd
pd.set_option('display.max_rows', None)

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import json
import shutil
import warnings
import re

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C
from scipy.stats import norm

warnings.filterwarnings('ignore')

# ============================================================================
# PROJECT PATHS (auto-detect repo root)
# ============================================================================

PROJECT_ROOT = Path.cwd()

# Allow the notebook to run from the repo root or from a nested notebook folder.
candidate_roots = [PROJECT_ROOT] + list(PROJECT_ROOT.parents[:3])

resolved_root = None
for candidate in candidate_roots:
    if (candidate / 'data' / 'initial_data').exists():
        resolved_root = candidate
        break

if resolved_root is None:
    resolved_root = PROJECT_ROOT

PROJECT_ROOT = resolved_root
DATA_ROOT = PROJECT_ROOT / 'data' / 'initial_data'
SUBMISSIONS_DIR = PROJECT_ROOT / 'data' / 'submissions'
PLOTS_DIR = PROJECT_ROOT / 'data' / 'plots'
LOGS_DIR = PROJECT_ROOT / 'data' / 'logs'

# ============================================================================
# CONSTANTS (Course Reference: Module 12 FAQ - Initial data sizes)
# ============================================================================

INITIAL_SIZES = {
    1: (10, 2),
    2: (10, 2),
    3: (15, 3),
    4: (30, 4),
    5: (20, 4),
    6: (20, 5),
    7: (30, 6),
    8: (40, 8),
}

FUNCTION_DESCRIPTIONS = {
    1: "2D source localisation (contamination/radiation detection)",
    2: "2D noisy scoring system with local optima",
    3: "3D drug discovery mixture optimisation",
    4: "4D business configuration with costly evaluations",
    5: "4D chemical process yield maximisation",
    6: "5D recipe-style optimisation",
    7: "6D hyperparameter tuning",
    8: "8D high-dimensional hyperparameter optimisation"
}

# ============================================================================
# BAYESIAN OPTIMISATION PARAMETERS
# ============================================================================

RNG_SEED = 42
N_CANDIDATES = 200_000
EI_XI = 0.001
UCB_KAPPA = 2.0

DEFAULT_TUNING = {
    "acq": "ei",
    "xi": EI_XI,
    "kappa": UCB_KAPPA,
    "candidate_pool": N_CANDIDATES,
    "n_restarts_optimizer": 10,
    "length_scale_bounds": (1e-2, 10.0),
    "noise_level": 1e-6,
}

WEEK8_TUNING = {
    1: {"acq": "ei",  "xi": 0.002, "kappa": 2.0, "candidate_pool": 120_000, "n_restarts_optimizer": 12, "length_scale_bounds": (1e-3, 8.0),  "noise_level": 1e-6},
    2: {"acq": "ei",  "xi": 0.003, "kappa": 2.0, "candidate_pool": 140_000, "n_restarts_optimizer": 12, "length_scale_bounds": (1e-3, 10.0), "noise_level": 1e-6},
    3: {"acq": "ei",  "xi": 0.008, "kappa": 2.2, "candidate_pool": 180_000, "n_restarts_optimizer": 14, "length_scale_bounds": (1e-3, 12.0), "noise_level": 1e-6},
    4: {"acq": "ucb", "xi": 0.010, "kappa": 1.8, "candidate_pool": 180_000, "n_restarts_optimizer": 14, "length_scale_bounds": (1e-3, 12.0), "noise_level": 1e-6},
    5: {"acq": "ei",  "xi": 0.001, "kappa": 2.0, "candidate_pool": 160_000, "n_restarts_optimizer": 14, "length_scale_bounds": (1e-3, 8.0),  "noise_level": 1e-6},
    6: {"acq": "ei",  "xi": 0.010, "kappa": 2.4, "candidate_pool": 220_000, "n_restarts_optimizer": 16, "length_scale_bounds": (1e-3, 15.0), "noise_level": 1e-6},
    7: {"acq": "ucb", "xi": 0.010, "kappa": 2.1, "candidate_pool": 260_000, "n_restarts_optimizer": 18, "length_scale_bounds": (1e-3, 20.0), "noise_level": 1e-5},
    8: {"acq": "ei",  "xi": 0.012, "kappa": 2.5, "candidate_pool": 300_000, "n_restarts_optimizer": 18, "length_scale_bounds": (1e-3, 25.0), "noise_level": 1e-5},
}

def get_bbo_tuning(func_id=None):
    """Return backward-compatible tuning config, with per-function overrides if func_id is provided."""
    cfg = dict(DEFAULT_TUNING)
    if func_id is not None:
        cfg.update(WEEK8_TUNING.get(int(func_id), {}))
    return cfg

def make_rng(seed_offset=0):
    return np.random.default_rng(RNG_SEED + int(seed_offset))

print("="*70)
print("BAYESIAN OPTIMISATION CAPSTONE - WEEK 7 READY WORKFLOW")
print("="*70)
print(f"Project Root: {PROJECT_ROOT}")
print(f"Data Root: {DATA_ROOT}")
print(f"Data exists: {DATA_ROOT.exists()}")
print(f"Submissions: {SUBMISSIONS_DIR.exists()}")
print("\nWeek 7 per-function tuning:")
for fid in range(1, 9):
    cfg = get_bbo_tuning(fid)
    print(
        f"  F{fid}: acq={cfg['acq'].upper():>3} | xi={cfg['xi']:.4f} | "
        f"kappa={cfg['kappa']:.1f} | candidates={cfg['candidate_pool']:,}"
    )
print("="*70)
print("✓ Configuration loaded")


## Cell 2: Directory Management

**Course note pointer:** Repository organisation requirements from Module 12 Mini-lesson 12.7

In [ ]:
"""
Directory Structure Validation
"""

def ensure_directories():
    """Create missing directories in the CapStone_BBO structure"""
    required_dirs = [
        SUBMISSIONS_DIR,
        PLOTS_DIR,
        PLOTS_DIR / 'current',
        PLOTS_DIR / 'historical',
        LOGS_DIR,
    ]
    
    created = []
    for dir_path in required_dirs:
        if not dir_path.exists():
            dir_path.mkdir(parents=True, exist_ok=True)
            created.append(dir_path.relative_to(PROJECT_ROOT))
    
    if created:
        print(f"✓ Created: {', '.join(str(d) for d in created)}")
    else:
        print("✓ All directories exist")
    
    # Create backup directories for each function
    if DATA_ROOT.exists():
        for k in range(1, 9):
            backup_dir = DATA_ROOT / f"function_{k}" / "backups"
            backup_dir.mkdir(exist_ok=True)

def validate_initial_data():
    """Validate initial data structure (Course Reference: FAQ - Initial data)"""
    if not DATA_ROOT.exists():
        print("❌ data/initial_data not found!")
        print("\nPlease make sure the live cumulative data folder exists at:")
        print(f"  {DATA_ROOT}")
        return False
    
    print("\nValidating initial data:")
    missing = []
    for k in range(1, 9):
        fdir = DATA_ROOT / f"function_{k}"
        inputs_file = fdir / "initial_inputs.npy"
        outputs_file = fdir / "initial_outputs.npy"
        
        if not inputs_file.exists() or not outputs_file.exists():
            missing.append(k)
        else:
            # Verify shape
            X = np.load(inputs_file)
            y = np.load(outputs_file)
            expected_shape = INITIAL_SIZES[k]
            status = "✓" if X.shape == expected_shape and len(y) == expected_shape[0] else "⚠"
            print(f"  {status} Function {k}: X{X.shape}, y{y.shape}")
    
    if missing:
        print(f"\n❌ Missing data for functions: {missing}")
        return False
    
    print("\n✓ All initial data validated")
    return True

# Run validation
ensure_directories()
validate_initial_data()

## Cell 3: Core BBO Functions - Portal Format Utilities

**Course note pointer:** Strict portal format from Module 12 FAQ - "What do I submit?"

In [ ]:
"""
Portal Format Utilities

Course Reference: Module 12 FAQ - Submission format requirements
- Format: x1-x2-x3-...-xn
- Exactly 6 decimals per value
- Hyphen-separated, no spaces
- Values in range [0.000000, 0.999999]
"""

def clamp01(x: np.ndarray) -> np.ndarray:
    """
    Clamp to valid capstone input range: [0.000000, 0.999999]
    
    Course Reference: FAQ - Input constraints
    """
    x = np.asarray(x, dtype=float)
    return np.clip(x, 0.0, 0.999999)

def portal_string(x: np.ndarray) -> str:
    """
    Format array to portal submission string: '0.123456-0.654321-...'
    
    Course Reference: FAQ - Exact formatting requirements
    """
    x = clamp01(x)
    return "-".join(f"{v:.6f}" for v in x)

def parse_portal_string(s: str, d: int) -> np.ndarray:
    """
    Parse portal string back to array
    """
    parts = s.strip().split("-")
    if len(parts) != d:
        raise ValueError(f"Expected {d} values, got {len(parts)}: {s!r}")
    return clamp01(np.array([float(p) for p in parts], dtype=float))

def validate_portal_string(s: str, d: int) -> bool:
    """
    Strict portal pattern validation: 0.xxxxxx with exactly six decimals
    
    Course Reference: FAQ - Common errors to avoid
    """
    pattern = r"^0\.\d{6}" + (r"-(0\.\d{6})" * (d - 1)) + r"$"
    return re.match(pattern, s.strip()) is not None

# Test the utilities
test_x = np.array([0.123456, 0.654321])
test_str = portal_string(test_x)
print(f"Example portal string: {test_str}")
print(f"Valid format: {validate_portal_string(test_str, 2)}")
print("✓ Portal format utilities loaded")

## Cell 4: Gaussian Process Surrogate Model

**Course note mapping (Module 12):**
- Surrogate model: Mini-lesson 12.1 - Gaussian Process Regression
- Kernel selection: Mini-lesson 12.2 - Matérn kernel for smoothness
- Hyperparameter optimisation: Mini-lesson 12.4 - Marginal likelihood

In [ ]:
"""
Gaussian Process Surrogate Model

Course Reference: Module 12 FAQ - "What optimisation method should I use?"
Recommended: Gaussian Process for expensive black-box functions
"""

def fit_gp_surrogate(X: np.ndarray, y: np.ndarray, func_id=None, random_state: int = 0):
    """
    Fit GP surrogate model with standardised outputs and function-specific tuning.

    Returns:
        gp: Fitted GaussianProcessRegressor
        y_mean: Mean used for standardisation
        y_std: Std used for standardisation
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float).ravel()

    if X.ndim != 2:
        raise ValueError(f"X must be 2D, got shape {X.shape}")
    if y.ndim != 1:
        raise ValueError(f"y must be 1D, got shape {y.shape}")
    if len(X) != len(y):
        raise ValueError(f"X/y length mismatch: {len(X)} vs {len(y)}")
    if not np.isfinite(X).all():
        raise ValueError("X contains NaN or inf")
    if not np.isfinite(y).all():
        raise ValueError("y contains NaN or inf")

    y_mean = float(y.mean())
    y_std = float(y.std()) if float(y.std()) > 0 else 1.0
    y_scaled = (y - y_mean) / y_std

    d = X.shape[1]
    cfg = get_bbo_tuning(func_id)
    ls_lo, ls_hi = cfg["length_scale_bounds"]
    noise_level = cfg["noise_level"]

    kernel = (
        C(1.0, (1e-3, 1e3))
        * Matern(length_scale=np.ones(d), length_scale_bounds=(ls_lo, ls_hi), nu=2.5)
        + WhiteKernel(noise_level=noise_level, noise_level_bounds=(1e-10, 1e-1))
    )

    gp = GaussianProcessRegressor(
        kernel=kernel,
        n_restarts_optimizer=cfg["n_restarts_optimizer"],
        random_state=random_state,
    )

    try:
        gp.fit(X, y_scaled)
    except Exception:
        gp = GaussianProcessRegressor(
            kernel=kernel,
            n_restarts_optimizer=2,
            random_state=random_state,
        )
        gp.fit(X, y_scaled)

    return gp, y_mean, y_std

print("✓ Gaussian Process surrogate model loaded")
print("  Kernel: Constant × Matérn(ν=2.5) + WhiteKernel")
print("  Hyperparameter optimisation: function-specific restarts")


## Cell 5: Acquisition Functions

**Course note mapping (Module 12):**
- Acquisition functions: Mini-lesson 12.6 - Trading off exploration and exploitation
- Expected Improvement: Mini-lesson 12.6 - Balance between mean and uncertainty
- Upper Confidence Bound: Mini-lesson 12.6 - Optimism in the face of uncertainty
- Probability of Improvement: Alternative acquisition strategy

In [ ]:
"""
Acquisition Functions for Bayesian Optimisation

Course Reference: Module 12 Mini-lesson 12.6 - Acquisition functions guide the search
"""

def acquisition_ei(Xcand: np.ndarray, gp: GaussianProcessRegressor, y_best: float, xi: float = EI_XI) -> np.ndarray:
    """
    Expected Improvement for maximisation
    
    Course Reference: Mini-lesson 12.6 - EI balances exploration (σ) and exploitation (μ)
    
    Args:
        Xcand: Candidate points
        gp: Fitted GP model
        y_best: Best observed value (scaled)
        xi: Exploration parameter (default: 0.01)
    """
    mu, sigma = gp.predict(Xcand, return_std=True)
    sigma = np.maximum(sigma, 1e-12)  # Numerical stability
    
    improvement = mu - y_best - xi
    Z = improvement / sigma
    
    # EI = E[max(0, improvement)] = improvement*Φ(Z) + σ*φ(Z)
    return improvement * norm.cdf(Z) + sigma * norm.pdf(Z)

def acquisition_ucb(Xcand: np.ndarray, gp: GaussianProcessRegressor, kappa: float = UCB_KAPPA) -> np.ndarray:
    """
    Upper Confidence Bound for maximisation
    
    Course Reference: Mini-lesson 12.6 - UCB = μ + κσ (optimism under uncertainty)
    
    Args:
        Xcand: Candidate points
        gp: Fitted GP model
        kappa: Exploration factor (default: 2.0)
    """
    mu, sigma = gp.predict(Xcand, return_std=True)
    return mu + kappa * sigma

def acquisition_pi(Xcand: np.ndarray, gp: GaussianProcessRegressor, y_best: float, xi: float = EI_XI) -> np.ndarray:
    """
    Probability of Improvement for maximisation
    
    Course Reference: Mini-lesson 12.6 - PI = P(f(x) > y_best + ξ)
    """
    mu, sigma = gp.predict(Xcand, return_std=True)
    sigma = np.maximum(sigma, 1e-12)
    Z = (mu - y_best - xi) / sigma
    return norm.cdf(Z)

def acquisition_maxvar(Xcand: np.ndarray, gp: GaussianProcessRegressor) -> np.ndarray:
    """
    Maximum Variance (pure exploration)
    
    Course Reference: Mini-lesson 12.6 - Explore where uncertainty is highest
    """
    _, sigma = gp.predict(Xcand, return_std=True)
    return sigma

print("✓ Acquisition functions loaded:")
print("  - Expected Improvement (EI) - default, balanced")
print("  - Upper Confidence Bound (UCB) - more exploratory")
print("  - Probability of Improvement (PI) - more exploitative")
print("  - Maximum Variance (MaxVar) - pure exploration")

## Cell 6: Proposal Generation

**Course note mapping (Module 12):**
- Iteratively select new points: FAQ - "Implement Bayesian optimisation"
- Acquisition function optimisation: Mini-lesson 12.6
- Avoiding duplicates: Best practice for robustness

In [ ]:

"""
Next Point Proposal via Bayesian Optimisation

Course Reference: Module 12 FAQ - "Iteratively select new points using acquisition function"
"""

def remove_duplicate_candidates(Xcand: np.ndarray, X_seen: np.ndarray, tol: float = 1e-12) -> np.ndarray:
    """Remove exact/near-exact candidates that have already been evaluated."""
    if X_seen is None or len(X_seen) == 0:
        return Xcand

    keep = []
    for x in Xcand:
        is_dup = np.any(np.all(np.isclose(X_seen, x, atol=tol, rtol=0.0), axis=1))
        if not is_dup:
            keep.append(x)

    if len(keep) == 0:
        return np.empty((0, Xcand.shape[1]))
    return np.asarray(keep)


def propose_next_point(
    X: np.ndarray,
    y: np.ndarray,
    func_id=None,
    acquisition: str = None,
    xi: float = None,
    kappa: float = None,
    n_candidates: int = None,
    rng: np.random.Generator = None,
):
    """
    Propose next evaluation point using Bayesian Optimisation.

    Backward-compatible. If func_id is provided, Week 8 settings are used
    unless explicitly overridden.
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float).ravel()
    
    
    if X.ndim != 2 or X.shape[0] == 0:
        raise ValueError("X must be a non-empty 2D array")

    if y.ndim != 1 or len(y) != X.shape[0]:
        raise ValueError("y must be 1D and match the number of rows in X")

    d = X.shape[1]

    cfg = get_bbo_tuning(func_id)
    acquisition = (acquisition or cfg["acq"]).lower().strip()
    xi = cfg["xi"] if xi is None else float(xi)
    kappa = cfg["kappa"] if kappa is None else float(kappa)
    n_candidates = int(cfg["candidate_pool"] if n_candidates is None else n_candidates)
    rng = make_rng(func_id or 0) if rng is None else rng

    try:
        gp, y_mean, y_std = fit_gp_surrogate(X, y, func_id=func_id, random_state=RNG_SEED)

        Xcand = clamp01(rng.random((n_candidates, d)))
        Xcand = remove_duplicate_candidates(Xcand, X)
        if len(Xcand) == 0:
            Xcand = clamp01(rng.random((max(10_000, d * 1000), d)))

        y_scaled = (y - y_mean) / y_std
        y_best = float(y_scaled.max())

        if acquisition == "ei":
            scores = acquisition_ei(Xcand, gp, y_best=y_best, xi=xi)
        elif acquisition == "ucb":
            scores = acquisition_ucb(Xcand, gp, kappa=kappa)
        elif acquisition == "pi":
            scores = acquisition_pi(Xcand, gp, y_best=y_best, xi=xi)
        elif acquisition in {"maxvar", "var", "variance"}:
            scores = acquisition_maxvar(Xcand, gp)
        else:
            raise ValueError("acquisition must be one of: 'ei', 'ucb', 'pi', 'maxvar'")

        idx = int(np.argmax(scores))
        x_next = clamp01(Xcand[idx])

        if np.any(np.all(np.isclose(X, x_next, atol=1e-12), axis=1)):
            x_next = clamp01(x_next + rng.normal(0, 1e-4, size=d))

        mu_next, sigma_next = gp.predict(x_next.reshape(1, -1), return_std=True)

        diagnostics = {
            "status": "ok",
            "func_id": func_id,
            "acquisition": acquisition,
            "xi": float(xi) if xi is not None else None,
            "kappa": float(kappa) if kappa is not None else None,
            "best_score": float(scores[idx]),
            "pred_mean_scaled": float(mu_next[0]),
            "pred_std_scaled": float(sigma_next[0]),
            "y_best_scaled": y_best,
            "y_best_original": float(y.max()),
            "n_candidates": int(len(Xcand)),
            "kernel": str(gp.kernel_),
        }

        return x_next, diagnostics

    except Exception as e:
        x_fallback = clamp01(rng.random(d))
        diagnostics = {
            "status": "fallback_random",
            "func_id": func_id,
            "acquisition": acquisition,
            "xi": float(xi) if xi is not None else None,
            "kappa": float(kappa) if kappa is not None else None,
            "n_candidates": int(n_candidates),
            "error": str(e),
        }
        return x_fallback, diagnostics
        
print("✓ Bayesian Optimisation proposal generator loaded")
print("  Function-specific Week 8 tuning active")


## Cell 7: Multi-Format Parser

**Course note pointer:** Portal provides multiple formats - this parser handles them all automatically

In [ ]:
"""
Multi-Format Portal File Parser

Handles: Portal confirmation, Numpy arrays, Email format, Clean format,
and processed-week archive files with per-week list rows.
"""

def parse_portal_confirmation(filepath):
    """Parse Week01-*.txt format"""
    with open(filepath, 'r') as f:
        content = f.read()

    inputs = []
    for line in content.split('\n'):
        line = line.strip()
        if '-' in line and not line.startswith('Function') and not line.startswith('Submission'):
            if line and line[0].isdigit():
                inputs.append(line)
    return inputs if len(inputs) == 8 else None

def parse_numpy_format(filepath):
    """Parse numpy array format"""
    with open(filepath, 'r') as f:
        content = f.read().strip()
    if content.startswith('[') and 'array' in content:
        try:
            arrays = eval(content, {"array": np.array, "np": np})
            return arrays
        except Exception:
            return None
    return None

def arrays_to_portal_format(arrays):
    """Convert arrays to portal strings"""
    return [portal_string(arr) for arr in arrays]

def load_week_inputs(week_num):
    """Load inputs for a week - try all formats"""
    week_dir = SUBMISSIONS_DIR / f"week_{week_num:02d}"
    if not week_dir.exists():
        return None, f"Week {week_num} directory not found"

    # Try portal confirmation first
    for conf_file in week_dir.glob("Week*.txt"):
        inputs = parse_portal_confirmation(conf_file)
        if inputs:
            return inputs, f"Portal confirmation: {conf_file.name}"

    # Try inputs.txt
    inputs_file = week_dir / "inputs.txt"
    if inputs_file.exists():
        # Try numpy format
        arrays = parse_numpy_format(inputs_file)
        if arrays:
            return arrays_to_portal_format(arrays), "Numpy format"
        # Try clean format
        with open(inputs_file, 'r') as f:
            lines = [line.strip() for line in f if line.strip() and not line.startswith('#')]
        if len(lines) == 8:
            return lines, "Clean format"

    return None, "No valid input file found"

def _try_parse_outputs_content(content, week_num=None):
    """
    Returns (outputs, format_label) or (None, reason).
    Supports:
    - 8 clean numeric lines
    - single python/numpy list of 8 numbers
    - multi-line archive file where each line is a list of 8 outputs for week 1..N
    """
    safe_globals = {"np": np, "array": np.array}

    # 1) Try whole-file eval first
    try:
        obj = eval(content, safe_globals)
        if isinstance(obj, (list, tuple, np.ndarray)):
            if len(obj) == 8 and all(np.isscalar(x) for x in obj):
                return [float(x) for x in obj], "Python/Numpy list format"
            if len(obj) >= 1 and all(isinstance(row, (list, tuple, np.ndarray)) and len(row) == 8 for row in obj):
                idx = (week_num - 1) if week_num is not None and 1 <= week_num <= len(obj) else len(obj) - 1
                return [float(x) for x in obj[idx]], f"Archive list-of-lists format (selected row {idx+1})"
    except Exception:
        pass

    # 2) Try line-by-line eval for archive files
    parsed_rows = []
    for line in content.splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        try:
            row = eval(line, safe_globals)
            if isinstance(row, (list, tuple, np.ndarray)) and len(row) == 8:
                parsed_rows.append([float(x) for x in row])
        except Exception:
            continue

    if parsed_rows:
        idx = (week_num - 1) if week_num is not None and 1 <= week_num <= len(parsed_rows) else len(parsed_rows) - 1
        return parsed_rows[idx], f"Archive row format (selected row {idx+1})"

    # 3) Fallback: clean numeric lines only
    clean_outputs = []
    for line in content.splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        try:
            clean_outputs.append(float(line))
        except Exception:
            continue

    if len(clean_outputs) == 8:
        return clean_outputs, "Clean line format"

    return None, f"Could not parse exactly 8 outputs (found {len(clean_outputs)} clean numeric values)."

def _find_processed_week_output_file(week_num):
    """
    Search project root for folders/files like:
    - Capstone Project – Week 06 Submission Processed/outputs.txt
    - Capstone Project - Week 06 Submission Processed/outputs.txt
    """
    patterns = [
        f"*Week {week_num:02d}*Processed*",
        f"*Week {week_num}*Processed*",
    ]

    candidates = []
    for pattern in patterns:
        candidates.extend((PROJECT_ROOT / 'data' / 'processed').glob(pattern))
        candidates.extend(PROJECT_ROOT.glob(pattern))

    for c in candidates:
        if c.is_dir():
            out = c / "outputs.txt"
            if out.exists():
                return out
            nested = list(c.glob("**/outputs.txt"))
            if nested:
                return nested[0]
        elif c.is_file() and c.name.lower() == "outputs.txt":
            return c

    return None

def load_week_outputs(week_num):
    """
    Load outputs for a week with flexible parsing.
    Fallbacks to processed-week archive folders if week_NN/outputs.txt is still a placeholder.
    """
    week_dir = SUBMISSIONS_DIR / f"week_{week_num:02d}"
    outputs_file = week_dir / "outputs.txt"

    if outputs_file.exists():
        content = outputs_file.read_text().strip()
        outputs, fmt = _try_parse_outputs_content(content, week_num=week_num)
        if outputs is not None:
            return outputs, fmt

    processed_file = _find_processed_week_output_file(week_num)
    if processed_file is not None:
        content = processed_file.read_text().strip()
        outputs, fmt = _try_parse_outputs_content(content, week_num=week_num)
        if outputs is not None:
            return outputs, f"{fmt} via processed archive: {processed_file}"

    if outputs_file.exists():
        return None, f"Could not parse valid outputs from {outputs_file}"
    return None, f"outputs.txt not found in {week_dir.name}, and no processed archive fallback found"

print("✓ Multi-format parser loaded (portal/numpy/clean/archive)")


## Cell 8: Data Management with Backups

**Course note mapping (Module 12):**
- Appending data: FAQ - Example code for appending new data points
- Data continuity: FAQ - "Keep track of all previously collected data points"

In [ ]:
"""
Data Management with Automatic Backups

Course Reference: Module 12 FAQ - Appending new data points workflow
"""

def get_current_data_state():
    """Get current state of all datasets"""
    state = {}
    for k in range(1, 9):
        y = np.load(DATA_ROOT / f"function_{k}" / "initial_outputs.npy")
        initial_size = INITIAL_SIZES[k][0]
        state[k] = {
            'total_points': len(y),
            'initial_size': initial_size,
            'weeks_added': len(y) - initial_size,
            'best_value': float(y.max()),
            'latest_value': float(y[-1]) if len(y) > initial_size else None
        }
    return state

def backup_function_data(k, week_num):
    """Create timestamped backup"""
    fdir = DATA_ROOT / f"function_{k}"
    backup_dir = fdir / "backups"
    backup_dir.mkdir(exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    X = np.load(fdir / "initial_inputs.npy")
    y = np.load(fdir / "initial_outputs.npy")
    
    np.save(backup_dir / f"inputs_before_week{week_num:02d}_{timestamp}.npy", X)
    np.save(backup_dir / f"outputs_before_week{week_num:02d}_{timestamp}.npy", y)
    return backup_dir

def append_week_data(week_inputs, week_outputs, week_num):
    """
    Append weekly results to all function datasets
    
    Course Reference: Module 12 FAQ - Example code pattern
    """
    print(f"\n{'='*70}")
    print(f"APPENDING WEEK {week_num} DATA")
    print(f"{'='*70}")
    
    summary = {}
    
    for k in range(1, 9):
        fdir = DATA_ROOT / f"function_{k}"
        
        # Load current data
        X = np.load(fdir / "initial_inputs.npy")
        y = np.load(fdir / "initial_outputs.npy")
        
        if X.shape[0] != len(y):
            raise ValueError(f"Function {k}: X rows {X.shape[0]} != y len {len(y)}")
        
        initial_n = INITIAL_SIZES[k][0]
        expected_before = initial_n + (week_num - 1)

        if len(y) != expected_before:
            raise ValueError(
                f"Refusing to append Week {week_num} for Function {k}: "
                f"current_size={len(y)} expected_before={expected_before}. "
                "Prevents duplicates/out-of-order appends."
        )

        
        # Parse new point
        x_new = parse_portal_string(week_inputs[k-1], d=X.shape[1])
        y_new = week_outputs[k-1]
        
        # Validate
        if len(x_new) != X.shape[1]:
            raise ValueError(f"Function {k}: Expected {X.shape[1]}D, got {len(x_new)}D")
        
        # Backup
        backup_function_data(k, week_num)
        
        # Append (Course Reference: FAQ pattern - vstack and append)
        X_updated = np.vstack((X, x_new))
        y_updated = np.append(y, y_new)
        
        # Save
        np.save(fdir / "initial_inputs.npy", X_updated)
        np.save(fdir / "initial_outputs.npy", y_updated)
        
        # Summary
        old_best = y.max()
        new_best = y_updated.max()
        improved = y_new > old_best
        
        summary[k] = {
            'old_size': len(y),
            'new_size': len(y_updated),
            'new_value': y_new,
            'old_best': old_best,
            'new_best': new_best,
            'improved': improved,
            'description': FUNCTION_DESCRIPTIONS[k]
        }
        
        status = "📈 NEW BEST" if improved else "📊"
        print(f"\nFunction {k} ({FUNCTION_DESCRIPTIONS[k]}):")
        print(f"  {status}")
        print(f"  Size: {len(y)} → {len(y_updated)}")
        print(f"  Input: {week_inputs[k-1]}")
        print(f"  Output: {y_new:+.6f}")
        print(f"  Best: {old_best:+.6f} → {new_best:+.6f}")
        if improved:
            print(f"  ✨ Improvement: {y_new - old_best:+.6f}")
    
    improvements = sum(1 for v in summary.values() if v['improved'])
    print(f"\n{'='*70}")
    print(f"✓ Week {week_num} appended: {improvements}/8 functions improved")
    print(f"{'='*70}")
    
    return summary

print("✓ Data management with automatic backups loaded")

## Cell 9: Visualisation (Inline Plots)

**Course note mapping (Module 12):**
- Visualise progress: FAQ - "Plot the progress of the optimisation process"
- Show improvement over time: FAQ - Requirement for final repository

In [ ]:
"""
Progress Visualisation with Inline Display

Course Reference: Module 12 FAQ - "Visualise progress" requirement
"""

def create_progress_plot(k, week_num=None, show_inline=True):
    """
    Create progress plot showing best-so-far convergence
    
    Course Reference: FAQ - Show optimisation progress over iterations
    """
    y = np.load(DATA_ROOT / f"function_{k}" / "initial_outputs.npy")
    bsf = np.maximum.accumulate(y)
    
    initial_size = INITIAL_SIZES[k][0]
    weeks_added = len(y) - initial_size
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(range(1, len(bsf) + 1), bsf, 'b-', linewidth=2.5, 
            label='Best so far', zorder=2)
    
    # Initial data boundary
    ax.axvline(x=initial_size, color='gray', linestyle='--', 
              label='Initial data', alpha=0.6, linewidth=1.5, zorder=1)
    
    # Weekly additions
    colours = ['red', 'green', 'orange', 'purple', 'brown', 'pink', 'cyan']
    for w in range(1, min(weeks_added + 1, 13)):
        idx = initial_size + w
        colour = colours[(w-1) % len(colours)]
        ax.plot(idx, bsf[idx-1], 'o', color=colour, markersize=10, 
               label=f'Week {w}', zorder=3)
        ax.axvline(x=idx, color=colour, linestyle=':', alpha=0.2, zorder=1)
    
    ax.set_xlabel("Evaluation Count", fontsize=13, fontweight='bold')
    ax.set_ylabel("Best-so-far Output", fontsize=13, fontweight='bold')
    
    title = f"Function {k}: {FUNCTION_DESCRIPTIONS[k]}\n"
    title += f"Best: {bsf[-1]:+.6f} ({weeks_added} week{'s' if weeks_added != 1 else ''})"
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(fontsize=9, loc='best', ncol=2)
    plt.tight_layout()
    
    # Save
    if week_num:
        save_dir = PLOTS_DIR / 'historical' / f"week_{week_num:02d}"
        save_dir.mkdir(parents=True, exist_ok=True)
        filename = f"function_{k}_week{week_num:02d}.png"
    else:
        save_dir = PLOTS_DIR / 'current'
        filename = f"function_{k}_current.png"
    
    save_path = save_dir / filename
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    
    if show_inline:
        plt.show()
    else:
        plt.close()
    
    return save_path

def generate_all_plots(week_num=None, show_inline=True):
    """Generate all progress plots with inline display"""
    print(f"\n{'='*70}")
    print("GENERATING PROGRESS PLOTS")
    print(f"{'='*70}\n")
    
    paths = []
    for k in range(1, 9):
        print(f"Function {k}: {FUNCTION_DESCRIPTIONS[k]}")
        path = create_progress_plot(k, week_num, show_inline=show_inline)
        paths.append(path)
        print(f"  ✓ Saved: {path.name}\n")
    
    save_location = PLOTS_DIR / ('historical' if week_num else 'current')
    print(f"{'='*70}")
    print(f"✓ All plots saved to: {save_location}")
    print(f"{'='*70}")
    return paths

print("✓ Visualisation functions loaded (with inline display)")

## Cell 10: Output Generation

**Course note pointer:** Repository structure requirements from Module 12 Mini-lesson 12.7

In [ ]:
"""
Output File Generation and Logging
"""

def save_proposals(proposals, next_week_num):
    """Save proposals in portal-ready format"""
    week_dir = SUBMISSIONS_DIR / f"week_{next_week_num:02d}"
    week_dir.mkdir(exist_ok=True)

    inputs_file = week_dir / "inputs.txt"
    with open(inputs_file, 'w') as f:
        for p in proposals:
            f.write(f"{p['portal_string']}\n")

    outputs_file = week_dir / "outputs.txt"
    with open(outputs_file, 'w') as f:
        f.write(f"# Week {next_week_num} Outputs\n")
        f.write(f"# Fill after portal submission\n")
        f.write(f"# One value per line (8 total)\n\n")

    guide_file = week_dir / "SUBMISSION_GUIDE.md"
    with open(guide_file, 'w') as f:
        f.write(f"# Week {next_week_num} Submission Guide\n\n")
        f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        f.write("## Portal Submission\n\n")
        f.write("Copy each line to the corresponding function in the portal:\n\n")

        for p in proposals:
            d = p['diagnostics']
            f.write(f"**Function {p['function']}** ({len(p['input_array'])}D):  \n")
            f.write(f"`{p['portal_string']}`\n\n")
            f.write(f"- Description: {p['description']}\n")
            f.write(f"- Current evaluations: {p['n_evaluations']}\n")
            f.write(f"- Current best: {p['current_best']:+.6f}\n")
            f.write(f"- Acquisition: {d.get('acquisition', 'n/a')}\n")
            f.write(f"- xi: {d.get('xi', 0.0):.4f}\n")
            f.write(f"- kappa: {d.get('kappa', 0.0):.4f}\n\n")

        f.write("## After Submission\n\n")
        f.write(f"1. Save portal confirmation as `Week{next_week_num:02d}_TIMESTAMP.txt`\n")
        f.write("2. Copy 8 output values to `outputs.txt`\n")
        f.write(f"3. Next processing step: enter week {next_week_num} in the notebook\n")
        f.write("4. Run workflow again\n")

    return inputs_file, outputs_file, guide_file

def save_execution_log(week_num, summary, proposals):
    """Save detailed execution log"""
    log_data = {
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'week_processed': week_num,
        'next_week': week_num + 1,
        'summary': summary,
        'proposals': proposals,
        'method': 'Gaussian Process with function-specific EI/UCB tuning',
        'settings': {
            'kernel': 'Constant × Matérn(ν=2.5) + WhiteKernel',
            'default_ei_xi': EI_XI,
            'default_ucb_kappa': UCB_KAPPA,
            'default_n_candidates': N_CANDIDATES,
            'week8_tuning': WEEK8_TUNING,
        }
    }

    def convert(obj):
        if isinstance(obj, (np.integer, np.floating)):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, dict):
            return {k: convert(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [convert(i) for i in obj]
        return obj

    log_data = convert(log_data)

    log_file = LOGS_DIR / f"week_{week_num:02d}_execution.json"
    with open(log_file, 'w') as f:
        json.dump(log_data, f, indent=2)

    return log_file

print("✓ Output generation functions loaded")


## Cell 11: Intelligent Workflow Orchestrator

**Course note pointer:** This implements the complete weekly workflow described in Module 12 Mini-lesson 12.7

In [ ]:
"""
Intelligent Workflow Orchestrator

Automatically:
- Detects current state
- Asks for week number
- Loads portal results
- Appends to datasets
- Generates next proposals
- Creates visualisations
- Saves all outputs

Course Reference: Module 12 Mini-lesson 12.7 - Complete weekly workflow
"""

def check_week_already_processed(week_num):
    """Check if week has already been processed"""
    state = get_current_data_state()
    for k, info in state.items():
        expected_size = INITIAL_SIZES[k][0] + week_num
        if info['total_points'] < expected_size:
            return False
    return True


def suggest_current_week():
    """Infer the latest fully processed week from dataset sizes."""
    state = get_current_data_state()
    weeks_added = [info['weeks_added'] for info in state.values()]
    return int(min(weeks_added))


def print_proposals_table(proposals):
    """Print proposals in formatted table"""
    print("\n" + "="*90)
    print("PORTAL SUBMISSION - COPY THESE VALUES")
    print("="*90)
    for p in proposals:
        d = p['diagnostics']
        print(f"\nFunction {p['function']}: {p['portal_string']}")
        print(f"  {p['description']}")
        print(
            f"  Current: {p['n_evaluations']} evals, best = {p['current_best']:+.6f} | "
            f"acq={d.get('acquisition', 'n/a').upper()} | xi={d.get('xi', 0.0):.4f} | "
            f"kappa={d.get('kappa', 0.0):.2f}"
        )
    print("\n" + "="*90)


def run_intelligent_workflow():
    """Main intelligent workflow - runs complete BBO cycle."""
    print("\n" + "="*70)
    print("BAYESIAN OPTIMISATION - INTELLIGENT WORKFLOW")
    print("="*70)

    ensure_directories()
    if not validate_initial_data():
        return

    print("\n" + "-"*70)
    print("CURRENT DATA STATE")
    print("-"*70)
    state = get_current_data_state()
    for k, info in state.items():
        print(f"Function {k}: {info['total_points']} points "
              f"({info['weeks_added']} weeks) | "
              f"Best: {info['best_value']:+.6f}")
    print("-"*70)

    suggested_week = suggest_current_week()
    print("\nWhich week's results do you want to process?")
    print("Example: enter 6 to process Week 6 results and generate Week 7 proposals.")
    print(f"Detected current processed week from data: {suggested_week}")

    try:
        raw = input(f"\nEnter week number [{suggested_week}]: ").strip()
        week_num = suggested_week if raw == "" else int(raw)
    except (ValueError, EOFError):
        print("❌ Invalid input")
        return

    if week_num < 1 or week_num > 12:
        print(f"❌ Week {week_num} outside range (1-12)")
        return

    next_week = week_num + 1

    print(f"\n{'='*70}")
    print(f"PROCESSING: Week {week_num} → Generating Week {next_week}")
    print(f"{'='*70}")

    already = check_week_already_processed(week_num)
    if already:
        response = input(f"\n⚠️  Week {week_num} appears processed. Continue? (y/n) [n]: ").strip().lower()
        if response != 'y':
            print("Cancelled")
            return
        print("✓ Week already processed: skipping append step to avoid duplicates.")
        skip_append = True
    else:
        skip_append = False

    print(f"\n[1/5] Loading Week {week_num} results...")
    inputs, input_format = load_week_inputs(week_num)
    if inputs is None:
        print(f"❌ {input_format}")
        print(f"Place files in: {SUBMISSIONS_DIR / f'week_{week_num:02d}'}")
        return
    print(f"✓ Loaded inputs ({input_format})")

    outputs, output_format = load_week_outputs(week_num)
    if outputs is None:
        print(f"❌ {output_format}")
        return
    print(f"✓ Loaded outputs ({output_format})")

    print(f"\n[2/5] Appending to datasets...")
    if not skip_append:
        try:
            summary = append_week_data(inputs, outputs, week_num)
        except Exception as e:
            print(f"❌ Error: {e}")
            return
    else:
        summary = {k: {"improved": False} for k in range(1, 9)}
        print("✓ Skipped append (already processed).")

    print(f"\n[3/5] Generating Week {next_week} proposals...")
    print("Method: Gaussian Process with function-specific EI/UCB tuning\n")

    proposals = []
    for k in range(1, 9):
        fdir = DATA_ROOT / f"function_{k}"
        X = np.load(fdir / "initial_inputs.npy")
        y = np.load(fdir / "initial_outputs.npy")
        cfg = get_bbo_tuning(k)

        x_next, diag = propose_next_point(
            X,
            y,
            func_id=k,
            acquisition=cfg["acq"],
            xi=cfg["xi"],
            kappa=cfg["kappa"],
            n_candidates=cfg["candidate_pool"],
        )
    
        # Guard: GP failed — warn if fallback is silently submitted
        if diag.get("status") == "fallback_random":
            print(f"⚠️  Function {k}: GP FAILED — random fallback used! Error: {diag['error']}")

        s = portal_string(x_next)
        if not validate_portal_string(s, d=X.shape[1]):
            print(f"⚠️  Function {k}: Portal validation failed")

        proposals.append({
            'function': k,
            'input_array': x_next,
            'portal_string': s,
            'current_best': float(y.max()),
            'n_evaluations': len(y),
            'description': FUNCTION_DESCRIPTIONS[k],
            'diagnostics': diag
        })

        print(f"Function {k} ({X.shape[1]}D): {s}")
        print(
            f"  {FUNCTION_DESCRIPTIONS[k]}\n"
            f"  Best: {y.max():+.6f}, Evals: {len(y)}, acq={cfg['acq'].upper()}, "
            f"xi={cfg['xi']:.4f}, kappa={cfg['kappa']:.2f}, candidates={cfg['candidate_pool']:,}"
        )

    print_proposals_table(proposals)

    print(f"\n[4/5] Creating visualisations...")
    try:
        generate_all_plots(week_num, show_inline=True)
    except Exception as e:
        print(f"⚠️  Warning: {e}")

    print(f"\n[5/5] Saving outputs...")
    try:
        inputs_file, outputs_file, guide_file = save_proposals(proposals, next_week)
        log_file = save_execution_log(week_num, summary, proposals)
        print(f"✓ Proposals: {inputs_file}")
        print(f"✓ Guide: {guide_file}")
        print(f"✓ Log: {log_file}")
    except Exception as e:
        print(f"⚠️  Warning: {e}")

    improvements = sum(1 for v in summary.values() if v['improved'])

    print("\n" + "="*70)
    print("WORKFLOW COMPLETE ✓")
    print("="*70)
    print(f"Week {week_num} processed: {improvements}/8 functions improved")
    print(f"Week {next_week} proposals ready")
    print("\nNext steps:")
    print(f"1. Submit proposals from: {inputs_file}")
    print(f"2. Save portal results to: {outputs_file}")
    print(f"3. Next time, run again with week={next_week}")
    print("="*70)

    return summary, proposals

print("="*70)
print("✓ INTELLIGENT WORKFLOW READY")
print("="*70)
print("Execute: run_intelligent_workflow()")
print("Tip: if your data already includes Week 6 results, press Enter to accept week 6 and generate Week 7.")
print("="*70)


In [ ]:
# Ensure plot directories exist without deleting historical output
PLOTS_DIR.mkdir(exist_ok=True)
(PLOTS_DIR / "current").mkdir(parents=True, exist_ok=True)
(PLOTS_DIR / "historical").mkdir(parents=True, exist_ok=True)

print("✓ Plot folders ready:")
print(f"  {PLOTS_DIR / 'current'}")
print(f"  {PLOTS_DIR / 'historical'}")


## Cell 12: RUN THE WORKFLOW

**Execute this cell to run the complete Bayesian Optimisation workflow**

In [ ]:
# RUN THIS TO START THE WORKFLOW
result = run_intelligent_workflow()

## Cell 13: Utility Functions (Optional)

**Course note pointer:** Additional helpers for data analysis and debugging

In [ ]:
"""
Utility Functions
"""

def show_current_state():
    """Display detailed current state"""
    print("\nCURRENT DATA STATE")
    print("="*70)
    state = get_current_data_state()
    for k, info in state.items():
        print(f"\nFunction {k} ({INITIAL_SIZES[k][1]}D): {FUNCTION_DESCRIPTIONS[k]}")
        print(f"  Total points: {info['total_points']} (initial: {info['initial_size']}, added: {info['weeks_added']})")
        print(f"  Best value: {info['best_value']:+.6f}")
        if info['latest_value'] is not None:
            print(f"  Latest value: {info['latest_value']:+.6f}")
    print("="*70)

def list_available_weeks():
    """List processed weeks"""
    weeks = sorted([d.name for d in SUBMISSIONS_DIR.glob("week_*") if d.is_dir()])
    print(f"\nProcessed weeks: {len(weeks)}")
    for w in weeks:
        wdir = SUBMISSIONS_DIR / w
        has_inputs = (wdir / "inputs.txt").exists() or len(list(wdir.glob("Week*.txt"))) > 0
        has_outputs = (wdir / "outputs.txt").exists()
        status = "✓" if has_inputs and has_outputs else "⚠"
        print(f"  {status} {w}")
    return weeks

def regenerate_plots(week_num=None, show_inline=True):
    """Regenerate all plots"""
    return generate_all_plots(week_num, show_inline)

# Uncomment to use:
# show_current_state()
# list_available_weeks()
# regenerate_plots()

print("✓ Utility functions loaded")